In [ ]:
!pip install gdown
import gdown

url = 'https://drive.google.com/drive/folders/1GZjxZhAQlmt_gHUcXogkVgeAWWNPcOqL?usp=sharing'
gdown.download_folder(url, quiet=True, use_cookies=False)

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import spacy
from transformers import AutoModel, AutoTokenizer, AutoConfig, DefaultDataCollator
from datasets import Dataset

nlp = spacy.load("en_core_web_sm")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Multi-Backbone MoE Architecture

In [ ]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, modelName, labelCount=2):
        # Load the baseline model
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelName).float()
        modelWidth = self.encoder.config.hidden_size
        num_logic_features = 4

        # Create experts which focus on different parts of the sentence
        self.semantic_expert = nn.Linear(modelWidth, modelWidth)
        self.entity_expert = nn.Linear(modelWidth, modelWidth)
        self.action_expert = nn.Linear(modelWidth, modelWidth)
        self.logic_expert = nn.Linear(num_logic_features, modelWidth)

        # Combines the results from the experts using a gating mechanism
        self.normalisationLayer = nn.LayerNorm(num_logic_features)
        self.gating = nn.Sequential(
            nn.Linear(modelWidth * 3 + num_logic_features, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Softmax(dim=-1)
        )

        self.classifier = nn.Linear(modelWidth, labelCount)
        self.lossFunc = nn.CrossEntropyLoss()

    # Encodes the input and extracts the [CLS] token representation
    def encode_cls(self, input_ids, attention_mask):
      outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
      cls_output = outputs.last_hidden_state[:, 0, :]
      return cls_output

    def forward(
        self,
        input_ids,
        attention_mask,
        entity_input_ids,
        entity_attention_mask,
        action_input_ids,
        action_attention_mask,
        logic_features=None,
        labels=None
    ):
        # Encode the main input, entity-focused input, and action-focused input
        main_cls = self.encode_cls(input_ids, attention_mask)
        entity_cls = self.encode_cls(entity_input_ids, entity_attention_mask)
        action_cls = self.encode_cls(action_input_ids, action_attention_mask)

        # If logic features are not provided, create a zero tensor
        if logic_features is None:
            logic_features = torch.zeros((input_ids.size(0), 4), device=input_ids.device)

        # Normalize logic features and pass through the logic expert
        logic_features = logic_features.float()
        norm_logic = self.normalisationLayer(logic_features)


        # Pass the CLS representations through their respective experts
        semanticRes = torch.tanh(self.semantic_expert(main_cls))
        entityRes = torch.tanh(self.entity_expert(entity_cls))
        actionRes = torch.tanh(self.action_expert(action_cls))
        logicRes = torch.tanh(self.logic_expert(norm_logic))


        # Use the gating mechanism to determine how much to rely on each expert
        gate_input = torch.cat([main_cls, entity_cls, action_cls, norm_logic], dim=-1)
        gate_weights = self.gating(gate_input)

        # Combine the expert outputs according to the gate weights
        experts = torch.stack([semanticRes, entityRes, actionRes, logicRes], dim=1)
        moe_output = torch.bmm(gate_weights.unsqueeze(1), experts).squeeze(1)
        
        logits = self.classifier(moe_output)

        loss = None
        if labels is not None:
            loss = self.lossFunc(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}




### Preprocessing and Feature Extraction

In [ ]:

# Extract words based on a specific POS
def get_pos_filtered_text(text, pos_tags):
    # Generate POS tags
    doc = nlp(str(text))
    # Only select those which have been specified
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    # Produce POS tags for both the premise and hypothesis
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    # Count how many negations we see between the 2 sentences - a mismatch indicates contradiction
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    # Removes punctuation and stopwords
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    # Determines the overlap between sentences using Jaccard Similarity
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]


# Factory to handle HuggingFace map function
def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)

        # Extract the relevant sentences for each expert
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
        
        # Encoders for the 2 experts
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
        
        # Returns the sentences which each expert relies on
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],

            "entity_input_ids": ent_enc["input_ids"],
            "entity_attention_mask": ent_enc["attention_mask"],

            "action_input_ids": act_enc["input_ids"],
            "action_attention_mask": act_enc["attention_mask"],

            "logic_features": [float(x) for x in extract_logic_features(example["premise"], example["hypothesis"])],
        }
    return preprocess


### Running Inference on Test Set

In [ ]:
def run_shared_task_inference(input_csv_path, output_csv_path):
    ensemble_path = "nli_ensemble_model"
    backbones = {
        "deberta": "microsoft/deberta-v3-small",
        "modernbert": "answerdotai/ModernBERT-base"
    }
    
    df = pd.read_csv(input_csv_path)
    all_model_logits = []

    # Loop through each model in the ensemble, load it, preprocess the data, and get predictions
    for name, path in backbones.items():
        specific_dir = os.path.join(ensemble_path, name)
        
        # Load the model and tokenizer, and prepare the dataset
        tokenizer = AutoTokenizer.from_pretrained(specific_dir)
        model = POSSpecializedMoE(path)
        model.load_state_dict(torch.load(os.path.join(specific_dir, "moe_weights.pt"), map_location=device))
        model.to(device).eval()
        
        # Preprocess the dataset using the same logic as during training
        prep_fn = make_preprocess_fn(tokenizer)
        ds = Dataset.from_pandas(df).map(prep_fn)
        
        loader = torch.utils.data.DataLoader(ds, batch_size=16, collate_fn=DefaultDataCollator())
        
        # Get predictions for the current model and store the logits for ensembling
        logits = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                logits.append(model(**batch)["logits"].cpu().numpy())
        
        all_model_logits.append(np.concatenate(logits, axis=0))

    # Average the logits from all models and determine the final predictions
    final_logits = np.mean(all_model_logits, axis=0)
    df['prediction'] = np.argmax(final_logits, axis=1)
    df[['prediction']].to_csv(output_csv_path, index=False)
    print(f"Submission saved to: {output_csv_path}")

run_shared_task_inference("test_data/NLI/test.csv", "Group_13_C.csv")